In [ ]:
import numpy as np
import scipy
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.metrics import classification_report

data_path = Path("") # Insert path to the folder containing the .npy files

In [ ]:
def align_eigenvector_signs(X_raw, num_features_per_vector):
    """
    Enforces sign consistency across eigenvectors to resolve the v == -v ambiguity.
    Flips each eigenvector so its maximum absolute value is always positive.
    """
    X_aligned = np.copy(X_raw)
    num_vectors = X_raw.shape[1] // num_features_per_vector
    
    for i in range(len(X_aligned)):
        for j in range(num_vectors):
            start_idx = j * num_features_per_vector
            end_idx = start_idx + num_features_per_vector
            vec = X_aligned[i, start_idx:end_idx]
            
            max_val_idx = np.argmax(np.abs(vec))
            X_aligned[i, start_idx:end_idx] = vec * np.sign(vec[max_val_idx])
            
    return X_aligned


def load_user_data():
    """
    Loads centroid data and cluster labels from specified files.
    """
    
    try:
        centroid_mat = scipy.io.loadmat(data_path / "<centroid_file>.mat")
        raw_centroids = centroid_mat["run_centroids"][0, :, :, :]
        run_centroids = np.concatenate([raw_centroids[:, 0, :, :], raw_centroids[:, 1, :, :]], axis=0)
        print("Loaded centroid data shape:", run_centroids.shape)
    except FileNotFoundError:
        print("Error: Centroid file not found!")

    try:
        cluster_labels = np.load(data_path / 'cluster_labels.npy')[0, :] 
        cluster_labels = np.concatenate([cluster_labels, cluster_labels], axis=0)
        cluster_labels = cluster_labels.astype(int) 
        print("Loaded cluster labels shape:", cluster_labels.shape)
    except FileNotFoundError:
        print("Error: Label file not found!")

    return run_centroids, cluster_labels


def get_eigen_details(cov):
    """
    Calculates eigenvalues and eigenvectors of the covariance matrix, sorted in descending order.
    """
    vals, vecs = np.linalg.eigh(cov)
    idx = np.argsort(vals)[::-1]
    return vals[idx], vecs[:, idx]

In [ ]:
# --- 1. Load Data & Conditional Splitting ---
data, labels = load_user_data()

# Convert labels to a numpy array for boolean masking
labels = np.array(labels)

# Create masks: Labels 1 and 3 are isolated exclusively for the holdout set
holdout_mask = (labels == 1) | (labels == 3)
train_mask = ~holdout_mask

# Split the dataset dynamically based on the masks
train_data = [data[idx] for idx in np.where(train_mask)[0]]
train_labels = labels[train_mask]

holdout_data = [data[idx] for idx in np.where(holdout_mask)[0]]
holdout_labels = labels[holdout_mask]

# 3 eigenvectors * 13 channels = 39 features total
CHANNELS_PER_VECTOR = 13 
X_raw = np.zeros((len(train_data), 39)) 
holdout_X_raw = np.zeros((len(holdout_data), 39))
y = np.zeros(len(train_labels))
holdout_y = np.zeros(len(holdout_labels))

# --- 2. Extract and Flatten Top 3 Eigenvectors Separately ---
for i, cov in enumerate(train_data):
    eigenvals, eigenvecs = get_eigen_details(cov)
    X_raw[i] = eigenvecs[:, :3].flatten() 

for i, cov in enumerate(holdout_data):
    eigvals_holdout, eigenvecs_holdout = get_eigen_details(cov)
    holdout_X_raw[i] = eigenvecs_holdout[:, :3].flatten()

# Map training labels to numerical classes
for i, label in enumerate(train_labels):
    if label == 1:                  # stable (Should be empty in train, kept for logical consistency)
        y[i] = 0
    elif label in [2, 3]:           # emergent
        y[i] = 1
    elif label == -1:               # noise / other
        y[i] = 2

# Map holdout labels to numerical classes
for i, label in enumerate(holdout_labels):
    if label == 1:                  # stable
        holdout_y[i] = 0
    elif label in [2, 3]:           # emergent
        holdout_y[i] = 1
    elif label == -1:               # noise / other
        holdout_y[i] = 2

# Run the sign alignment to resolve v == -v subspace orientation ambiguity
X = align_eigenvector_signs(X_raw, num_features_per_vector=CHANNELS_PER_VECTOR)
holdout_X = align_eigenvector_signs(holdout_X_raw, num_features_per_vector=CHANNELS_PER_VECTOR)

print("Train Data shape after sign alignment:", X.shape)
print("Train Label Shape:", y.shape)
print("Holdout Data Shape:", holdout_X.shape)
print("Holdout Label Shape:", holdout_y.shape)

# --- 3. Base Evaluation ---
classifier = LinearDiscriminantAnalysis()
cv_scheme = StratifiedKFold(n_splits=5, shuffle=True, random_state=19)

print("\nRunning Permutation Test (500 shuffles)...")
score, permutation_scores, pvalue = permutation_test_score(
    classifier, X, y, cv=cv_scheme, n_permutations=500, n_jobs=-1, random_state=42
)

# --- 4. CV Partition Sensitivity Analysis ---
print("Running CV Split Sensitivity Analysis...")
sensitivity_scores = []
seeds = [10, 23, 42, 57, 89, 101, 144, 202, 305, 999]

for seed in seeds:
    sens_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_scores = []
    for train_idx, test_idx in sens_cv.split(X, y):
        classifier.fit(X[train_idx], y[train_idx])
        fold_scores.append(classifier.score(X[test_idx], y[test_idx]))
    sensitivity_scores.append(np.mean(fold_scores))

# --- 5. Print Validation and Sensitivity Report ---
print("\n================ VALIDATION & SENSITIVITY REPORT ================")
print(f"Observed Classification Accuracy: {score * 100:.2f}%")
print(f"Empirical p-value: {pvalue:.4f}")
print(f"Mean Accuracy across 10 random seeds: {np.mean(sensitivity_scores) * 100:.2f}%")
print(f"Accuracy Standard Deviation across seeds: {np.std(sensitivity_scores) * 100:.2f}%")
if np.std(sensitivity_scores) < 0.03:
    print("-> Sensitivity Check: PASSED (Model performance is highly stable across splits).")
else:
    print("-> Sensitivity Check: WARNING (Model performance fluctuates based on data splits).")
print("=================================================================\n")

# Generate out-of-fold predictions for final training metrics
oof_predictions = np.zeros_like(y)
for train_idx, test_idx in cv_scheme.split(X, y):
    classifier.fit(X[train_idx], y[train_idx])
    oof_predictions[test_idx] = classifier.predict(X[test_idx])

print("Detailed Training Classification Performance:")
# Dynamically adjust target names if a class is entirely missing from the training partition
unique_train_classes = np.unique(y).astype(int)
all_target_names = ['Stable', 'Emergent', 'Other']
active_target_names = [all_target_names[idx] for idx in unique_train_classes]

print(classification_report(y, oof_predictions, target_names=active_target_names))

# --- 6. Evaluate on Holdout Set & Extract Specific Run Statistics ---
classifier.fit(X, y)
holdout_predictions = classifier.predict(holdout_X)

print("\nHoldout Set Performance:")
print(f"Holdout Accuracy: {classifier.score(holdout_X, holdout_y) * 100:.2f}%")
print(classification_report(holdout_y, holdout_predictions, target_names=['Stable', 'Emergent', 'Other']))

print("\n================ SPECIFIC TRACKING STATISTICS ================")
print("Distribution of Holdout Classifications grouped by Original Labels 1 and 3:")

target_names_map = {0: 'Stable', 1: 'Emergent', 2: 'Other'}

for targeted_orig_label in [1, 3]:
    matching_indices = np.where(holdout_labels == targeted_orig_label)[0]
    total_runs = len(matching_indices)
    
    print(f"\nOriginal Cluster Label {targeted_orig_label} (Total Holdout Runs: {total_runs}):")
    if total_runs > 0:
        sub_group_preds = holdout_predictions[matching_indices]
        counts = np.bincount(sub_group_preds.astype(int), minlength=3)
        
        for numeric_class, label_name in target_names_map.items():
            freq = counts[numeric_class]
            percentage = (freq / total_runs) * 100
            print(f"  Classified as '{label_name}': {freq} runs ({percentage:.2f}%)")
    else:
        print("  No runs corresponding to this label were found in the holdout matrix.")
print("===============================================================")